# 🎬 Entrenador del Director IA (Unsloth + LLaMA 3)
Esta libreta está configurada con las versiones exactas para evitar errores de compatibilidad.
Solo necesitas correr las celdas una por una.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. Instalar dependencias actualizadas
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = True

# 2. Cargar el modelo base LLaMA 3 8B Instruct
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# 3. Configurar LoRA (Los adaptadores que se entrenarán)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
# 4. Formato de Prompt (Adaptado para Frutinovela sin campo input)
alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}```"""

EOS_TOKEN = tokenizer.eos_token # Token de fin de secuencia
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = alpaca_prompt.format(instruction, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

In [ ]:
# 5. Cargar Dataset (Asegúrate de que dataset_director.jsonl esté en Drive)
from datasets import load_dataset
dataset = load_dataset("json", data_files="/content/drive/MyDrive/modelo_director_ia/dataset_director.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True,)
print(f"\nDataset cargado: {len(dataset)} ejemplos listos para entrenar.")

In [ ]:
# 6. Configurar el Entrenador (SFTTrainer)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Ajusta los pasos según tu dataset
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

In [ ]:
# 7. 🚀 ¡INICIAR ENTRENAMIENTO!
trainer_stats = trainer.train()

In [ ]:
# 8. Guardar modelo en formato GGUF en Google Drive
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if True: model.save_pretrained_gguf("/content/drive/MyDrive/modelo_director_ia/director_ia_model", tokenizer, quantization_method = "q8_0")
print("\n✅ ¡Entrenamiento finalizado y modelo GGUF guardado en Google Drive!")